# Day06下午个人项目：电商用户数据可视化

姓名/学号或GitHub用户名：**24012462**  
第5天专题（A/B/C/D/E）：**A**

本Notebook需要完成4张独立图、1张综合图和1份图表清单。请阅读`docs/day06_student_visualization_manual.md`后开始。


## 项目规则

1. 使用第4天清洗数据，并核对第5天个人分析结果；
2. 柱状图和散点图必做；折线图只能用于时间或有序阶段；
3. 饼图只用于少量类别的整体构成，必要时改用柱状图；
4. 每张图写“观察—证据—边界”；
5. 输出文件名和目录不得修改，以便第7天Flask直接复用。


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

STUDENT_ID = "24012462"
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "PingFang SC",
    "Heiti SC", "Arial Unicode MS", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False


def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "day04_ecommerce_customer_cleaned.xlsx").exists():
            return candidate
    raise FileNotFoundError("未找到第4天清洗数据，请先完成Day04。")


ROOT = find_workspace_root()
DATA_PATH = ROOT / "day04_ecommerce_customer_cleaned.xlsx"
DAY05_DIR = ROOT
OUTPUT_DIR = ROOT / "output" / "day06_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("学生：", STUDENT_ID)
print("专题：", TOPIC)
print("输出：", OUTPUT_DIR.relative_to(ROOT))


## 检查点1：输入与业务问题

先验证4个输入文件，再写出4个问题。不要在尚未理解指标时直接绘图。


In [ ]:
required_inputs = [
    DATA_PATH,
    DAY05_DIR / "day05_overall_metrics.csv",
    DAY05_DIR / "day05_segment_analysis.csv",
    DAY05_DIR / "day05_cross_analysis.csv",
]
missing_inputs = [str(path.relative_to(ROOT)) for path in required_inputs if not path.exists()]
assert not missing_inputs, f"缺少输入文件：{missing_inputs}"

df = pd.read_excel(DATA_PATH)
df['TenureGroup'] = pd.cut(df['Tenure'], bins=[-1, 0, 6, 12, 24, 999], labels=['新用户', '0-6个月', '7-12个月', '13-24个月', '24个月以上'])
overall_metrics = pd.read_csv(required_inputs[1])
segment_analysis = pd.read_csv(required_inputs[2])
cross_analysis = pd.read_csv(required_inputs[3])

assert df.shape[0] == 5630, f"清洗数据行数异常：{df.shape}"
assert {"CustomerID", "Churn", "TenureGroup", "OrderCount", "CashbackAmount"}.issubset(df.columns)
assert set(df["Churn"].dropna().unique()).issubset({0, 1})

display(overall_metrics)
display(segment_analysis.head())
display(cross_analysis.head())
print("检查点1A通过：输入文件有效")


In [ ]:
# 填写4个业务问题和图表选择理由
business_questions = {
    "category_bar": "不同城市等级的用户流失率有何差异？",
    "behavior_scatter": "用户订单数与返现金额之间存在怎样的关系？",
    "ordered_line": "不同使用时长阶段的用户流失率呈现怎样的变化趋势？",
    "composition_chart": "用户满意度评分的整体构成分布如何？",
}

chart_reasons = {
    "category_bar": "城市等级是离散分类变量，适合用柱状图展示不同类别的流失率差异，并标注样本量",
    "behavior_scatter": "订单数和返现金额都是连续数值变量，散点图可以直观展示两者的相关性及流失用户分布",
    "ordered_line": "TenureGroup是有序阶段变量（新用户→0-6个月→7-12个月→13-24个月→24个月以上），折线图适合展示趋势变化",
    "composition_chart": "满意度评分只有5个类别（1-5分），属于少量类别，适合用饼图展示整体构成",
}

assert all(text.strip() for text in business_questions.values()), "请填写4个业务问题"
assert all(text.strip() for text in chart_reasons.values()), "请填写4个图表选择理由"
print("检查点1B通过：业务问题和选择理由已填写")


## 任务1：类别比较柱状图

要求：选择一个离散分组字段，计算用户数和一个核心指标；若绘制比率，标签中必须同时给出样本量。


In [ ]:
# 完成绘图数据
category_field = "CityTier"
category_summary = (
    df.groupby(category_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)
category_summary["CityTier"] = category_summary["CityTier"].astype(int).astype(str) + "线城市"

assert category_field in df.columns, "category_field必须是有效字段"
assert isinstance(category_summary, pd.DataFrame), "category_summary必须是DataFrame"
assert {category_field, "用户数"}.issubset(category_summary.columns)
display(category_summary)


In [ ]:
# 绘制并保存柱状图
fig_bar, ax_bar = plt.subplots(figsize=(10, 6))

bars = ax_bar.bar(category_summary["CityTier"], category_summary["流失率"], color="#4A90D9")
ax_bar.set_title("不同城市等级用户流失率对比", fontsize=14, fontweight="bold")
ax_bar.set_xlabel("城市等级", fontsize=12)
ax_bar.set_ylabel("流失率", fontsize=12)
ax_bar.yaxis.set_major_formatter(PercentFormatter(1))

for i, bar in enumerate(bars):
    height = bar.get_height()
    ax_bar.text(bar.get_x() + bar.get_width() / 2., height,
                f"{height:.1%}\n(n={category_summary['用户数'][i]})",
                ha="center", va="bottom", fontsize=10)

plt.grid(axis="y", alpha=0.3)
bar_path = OUTPUT_DIR / "01_category_bar.png"
fig_bar.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()

assert bar_path.exists() and bar_path.stat().st_size > 0, "柱状图尚未保存"
print("已输出：", bar_path.relative_to(ROOT))


### 柱状图结论

- 观察：一线城市用户流失率最低，二线城市流失率最高。
- 证据：一线城市用户3466人，流失率15.5%；二线城市用户242人，流失率34.3%；三线城市用户1922人，流失率17.1%。二线城市流失率显著高于一、三线城市。
- 边界：该图仅展示城市等级与流失率的相关性，无法证明因果关系；二线城市样本量较小（242人），结论稳定性需注意。


## 任务2：用户行为散点图

要求：选择两个数值字段，一行代表一个用户，颜色区分`Churn`，设置透明度。


In [ ]:
# 选择两个数值字段
x_field = "OrderCount"
y_field = "CashbackAmount"

assert x_field in df.columns and y_field in df.columns
assert pd.api.types.is_numeric_dtype(df[x_field])
assert pd.api.types.is_numeric_dtype(df[y_field])

fig_scatter, ax_scatter = plt.subplots(figsize=(10, 6))

churn_0 = df[df["Churn"] == 0]
churn_1 = df[df["Churn"] == 1]

ax_scatter.scatter(churn_0[x_field], churn_0[y_field], alpha=0.5, label="未流失", color="#4A90D9", s=30)
ax_scatter.scatter(churn_1[x_field], churn_1[y_field], alpha=0.5, label="已流失", color="#E53935", s=30)

ax_scatter.set_title("用户订单数与返现金额关系图", fontsize=14, fontweight="bold")
ax_scatter.set_xlabel("订单数", fontsize=12)
ax_scatter.set_ylabel("返现金额（元）", fontsize=12)
ax_scatter.legend(fontsize=12)
plt.grid(True, alpha=0.3)

scatter_path = OUTPUT_DIR / "02_behavior_scatter.png"
fig_scatter.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.show()

assert scatter_path.exists() and scatter_path.stat().st_size > 0, "散点图尚未保存"
print("已输出：", scatter_path.relative_to(ROOT))


### 散点图结论

- 观察：订单数与返现金额呈现正相关关系；已流失用户多集中在低订单数、低返现金额区域。
- 证据：订单数较少（1-3单）且返现金额较低（100-200元）的用户中，已流失用户占比较高；随着订单数和返现金额增加，未流失用户比例明显上升。
- 边界：相关关系不等于因果关系，不能确定是低返现导致流失，还是流失导致返现减少。


## 任务3：有序阶段折线图

当前数据没有日期。建议使用`TenureGroup`或`SatisfactionScore`，并明确写成“阶段比较”。


In [ ]:
TENURE_ORDER = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]

# 准备有序绘图数据
ordered_field = "TenureGroup"
ordered_summary = (
    df.groupby(ordered_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reindex(TENURE_ORDER)
      .reset_index()
)

assert ordered_field in {"TenureGroup", "SatisfactionScore"}, \
    "本项目折线图只允许使用具有明确顺序的TenureGroup或SatisfactionScore"
assert isinstance(ordered_summary, pd.DataFrame)
assert {ordered_field, "用户数"}.issubset(ordered_summary.columns)
display(ordered_summary)


In [ ]:
# 绘制折线图
fig_line, ax_line = plt.subplots(figsize=(10, 6))

ax_line.plot(ordered_summary[ordered_field], ordered_summary["流失率"], marker="o", 
             linewidth=2, color="#4A90D9", label="流失率")

ax_line.set_title("不同使用时长阶段用户流失率趋势", fontsize=14, fontweight="bold")
ax_line.set_xlabel("使用时长阶段", fontsize=12)
ax_line.set_ylabel("流失率", fontsize=12)
ax_line.yaxis.set_major_formatter(PercentFormatter(1))

for i, (rate, count) in enumerate(zip(ordered_summary["流失率"], ordered_summary["用户数"])):
    ax_line.annotate(f"{rate:.1%}\n(n={count})",
                     (ordered_summary[ordered_field][i], rate),
                     textcoords="offset points", xytext=(0, 10), ha="center", fontsize=10)

plt.grid(True, alpha=0.3)
ax_line.legend(fontsize=12)

line_path = OUTPUT_DIR / "03_ordered_line.png"
fig_line.savefig(line_path, dpi=150, bbox_inches="tight")
plt.show()

assert line_path.exists() and line_path.stat().st_size > 0, "折线图尚未保存"
print("已输出：", line_path.relative_to(ROOT))


### 折线图结论

- 观察：用户流失率随使用时长增加呈显著下降趋势，新用户流失率最高，24个月以上用户流失率为0。
- 证据：新用户流失率53.5%（508人），0-6个月25.9%（1642人），7-12个月9.8%（1584人），13-24个月6.5%（1467人），24个月以上0%（429人）。
- 边界：这是有序阶段比较，不是月度、年度或历史时间趋势；24个月以上用户样本量较小（429人），结论需谨慎解读。


## 任务4：整体构成图

类别少于或等于5个时可以使用饼图或环形图；否则改用柱状图。必须在选择理由中说明判断。


In [ ]:
# 选择构成字段并准备汇总表
composition_field = "SatisfactionScore"
composition_summary = (
    df.groupby(composition_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"))
      .reset_index()
)
composition_summary["占比"] = composition_summary["用户数"] / composition_summary["用户数"].sum()
composition_summary["SatisfactionScore"] = "满意度" + composition_summary["SatisfactionScore"].astype(str) + "分"

assert composition_field in df.columns
assert isinstance(composition_summary, pd.DataFrame)
assert {composition_field, "用户数", "占比"}.issubset(composition_summary.columns)
assert np.isclose(composition_summary["占比"].sum(), 1.0), "构成占比之和应为1"
display(composition_summary)


In [ ]:
# 类别不超过5个时绘制环形图
fig_composition, ax_composition = plt.subplots(figsize=(10, 6))

colors = ["#4A90D9", "#66BB6A", "#FFA726", "#EF5350", "#AB47BC"]
wedges, texts, autotexts = ax_composition.pie(
    composition_summary["占比"],
    labels=composition_summary["SatisfactionScore"],
    autopct="%1.1f%%",
    startangle=90,
    colors=colors,
    wedgeprops={"width": 0.4},
    textprops={"fontsize": 11}
)

ax_composition.set_title("用户满意度评分构成分布", fontsize=14, fontweight="bold")

legend_labels = [f"{label} (n={count})" for label, count in \
                 zip(composition_summary["SatisfactionScore"], composition_summary["用户数"])]
ax_composition.legend(wedges, legend_labels, loc="upper right", bbox_to_anchor=(1.3, 1))

composition_path = OUTPUT_DIR / "04_composition_chart.png"
fig_composition.savefig(composition_path, dpi=150, bbox_inches="tight")
plt.show()

assert composition_path.exists() and composition_path.stat().st_size > 0, "构成图尚未保存"
print("已输出：", composition_path.relative_to(ROOT))


### 构成图结论

- 观察：用户满意度分布较为均匀，3分和4分占比最高，1分最低。
- 证据：满意度3分占比24.4%（1375人），4分占比22.5%（1267人），2分占比19.8%（1116人），5分占比17.6%（992人），1分占比15.7%（880人）。
- 边界：该图适合展示整体构成比例，但不适合进行类别间的精确数值比较；满意度评分是主观指标，可能存在偏差。


## 检查点2与3：基础图表、优化和解释

逐项使用`docs/day06_chart_checklist.md`检查。确认比率图给出样本量、中文正常、颜色含义一致。


In [ ]:
individual_paths = [bar_path, scatter_path, line_path, composition_path]
for path in individual_paths:
    assert path.exists() and path.suffix.lower() == ".png"
    assert path.stat().st_size > 5_000, f"图片可能为空或质量过低：{path.name}"

print("检查点2通过：4张独立图已生成")
print("检查点3需要结合图表和文字结论人工复核")


## 任务5：2×2综合图

重新在4个子图中绘制核心内容，不要把4张PNG作为截图拼接。统一标题、颜色、字体和留白。


In [ ]:
fig_summary, axes = plt.subplots(2, 2, figsize=(14, 10))

# 子图1：柱状图 - 城市等级流失率
ax1 = axes[0, 0]
bars1 = ax1.bar(category_summary["CityTier"], category_summary["流失率"], color="#4A90D9")
ax1.set_title("(a) 城市等级流失率对比", fontsize=12, fontweight="bold")
ax1.set_xlabel("城市等级", fontsize=10)
ax1.set_ylabel("流失率", fontsize=10)
ax1.yaxis.set_major_formatter(PercentFormatter(1))
for i, bar in enumerate(bars1):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width() / 2., height,
             f"{height:.0%}", ha="center", va="bottom", fontsize=8)
ax1.grid(axis="y", alpha=0.3)

# 子图2：散点图 - 订单数与返现金额
ax2 = axes[0, 1]
ax2.scatter(churn_0[x_field], churn_0[y_field], alpha=0.4, label="未流失", color="#4A90D9", s=20)
ax2.scatter(churn_1[x_field], churn_1[y_field], alpha=0.4, label="已流失", color="#E53935", s=20)
ax2.set_title("(b) 订单数与返现金额关系", fontsize=12, fontweight="bold")
ax2.set_xlabel("订单数", fontsize=10)
ax2.set_ylabel("返现金额（元）", fontsize=10)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# 子图3：折线图 - 使用时长阶段流失率
ax3 = axes[1, 0]
ax3.plot(ordered_summary[ordered_field], ordered_summary["流失率"], marker="o", 
         linewidth=2, color="#4A90D9")
ax3.set_title("(c) 使用时长阶段流失率趋势", fontsize=12, fontweight="bold")
ax3.set_xlabel("使用时长阶段", fontsize=10)
ax3.set_ylabel("流失率", fontsize=10)
ax3.yaxis.set_major_formatter(PercentFormatter(1))
for i, rate in enumerate(ordered_summary["流失率"]):
    ax3.annotate(f"{rate:.0%}", (ordered_summary[ordered_field][i], rate),
                 textcoords="offset points", xytext=(0, 5), ha="center", fontsize=8)
ax3.grid(True, alpha=0.3)

# 子图4：环形图 - 满意度构成
ax4 = axes[1, 1]
colors = ["#4A90D9", "#66BB6A", "#FFA726", "#EF5350", "#AB47BC"]
ax4.pie(composition_summary["占比"], labels=composition_summary["SatisfactionScore"],
        autopct="%1.0f%%", startangle=90, colors=colors, wedgeprops={"width": 0.4},
        textprops={"fontsize": 8})
ax4.set_title("(d) 用户满意度构成分布", fontsize=12, fontweight="bold")

fig_summary.suptitle("电商用户数据可视化分析概览", fontsize=16, fontweight="bold")
fig_summary.tight_layout(rect=[0, 0, 1, 0.96])

summary_path = OUTPUT_DIR / "day06_visualization_summary.png"
fig_summary.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.show()

assert summary_path.exists() and summary_path.stat().st_size > 0, "综合图尚未保存"
print("已输出：", summary_path.relative_to(ROOT))


## 综合发现与局限

1. 综合发现1：用户生命周期是影响流失率的最关键因素。新用户流失率高达53.5%，而24个月以上用户流失率为0，说明随着用户使用时间增长，留存率显著提升。
2. 综合发现2：城市等级与流失率存在关联。二线城市用户流失率（34.3%）显著高于一线城市（15.5%）和三线城市（17.1%），可能与二线城市竞争环境或用户期望有关。
3. 综合发现3：用户活跃度（订单数）与返现金额呈正相关，且高订单数用户流失率更低。这表明提升用户购买频次和返现激励有助于降低流失率。
4. 数据或方法局限：数据中无时间序列信息，无法分析流失率的时间变化趋势；CashbackAmount是返现金额而非销售额，不能直接反映用户价值；部分细分群体（如二线城市）样本量较小，结论稳定性有限。


## 任务6：图表清单与检查点4

清单是第7天Flask读取图表说明的基础。每张图填写业务问题、图表类型、主要发现和局限。


In [ ]:
# 填写5行清单
chart_manifest = pd.DataFrame([
    {"chart_id": "01", "file_name": "01_category_bar.png", "business_question": "不同城市等级的用户流失率有何差异？", "chart_type": "bar", "key_finding": "二线城市用户流失率最高（34.3%），一线城市最低（15.5%）", "limitation": "二线城市样本量较小，结论稳定性需注意；仅展示相关性，无法证明因果"},
    {"chart_id": "02", "file_name": "02_behavior_scatter.png", "business_question": "用户订单数与返现金额之间存在怎样的关系？", "chart_type": "scatter", "key_finding": "订单数与返现金额呈正相关，已流失用户多集中在低订单数、低返现区域", "limitation": "相关关系不等于因果关系；数据存在重叠，部分细节不够清晰"},
    {"chart_id": "03", "file_name": "03_ordered_line.png", "business_question": "不同使用时长阶段的用户流失率呈现怎样的变化趋势？", "chart_type": "line", "key_finding": "流失率随使用时长增加显著下降，新用户流失率53.5%，24个月以上为0%", "limitation": "这是有序阶段比较，不是时间趋势；24个月以上样本量较小"},
    {"chart_id": "04", "file_name": "04_composition_chart.png", "business_question": "用户满意度评分的整体构成分布如何？", "chart_type": "pie", "key_finding": "满意度分布较为均匀，3分和4分占比最高（合计46.9%）", "limitation": "不适合进行类别间精确数值比较；满意度是主观指标，可能存在偏差"},
    {"chart_id": "05", "file_name": "day06_visualization_summary.png", "business_question": "整体概览", "chart_type": "dashboard", "key_finding": "综合展示了城市等级、用户行为、使用时长和满意度四个维度的流失相关特征", "limitation": "子图空间有限，细节信息展示不够充分"},
])

assert len(chart_manifest) == 5
assert not chart_manifest.astype(str).apply(lambda col: col.str.contains("请填写").any()).any(), \
    "请完成图表清单"

manifest_path = OUTPUT_DIR / "chart_manifest.csv"
chart_manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(chart_manifest)


In [ ]:
required_outputs = [
    OUTPUT_DIR / "01_category_bar.png",
    OUTPUT_DIR / "02_behavior_scatter.png",
    OUTPUT_DIR / "03_ordered_line.png",
    OUTPUT_DIR / "04_composition_chart.png",
    OUTPUT_DIR / "day06_visualization_summary.png",
    OUTPUT_DIR / "chart_manifest.csv",
]
missing_outputs = [str(path.relative_to(ROOT)) for path in required_outputs if not path.exists()]
assert not missing_outputs, f"缺少成果文件：{missing_outputs}"

manifest_check = pd.read_csv(OUTPUT_DIR / "chart_manifest.csv")
assert list(manifest_check.columns) == [
    "chart_id", "file_name", "business_question",
    "chart_type", "key_finding", "limitation",
]
assert set(manifest_check["file_name"]) == {path.name for path in required_outputs[:-1]}

print("检查点4通过：第6天成果物完整")
print("下一步：重启内核并从头运行，然后执行提交检查脚本并推送GitHub。")
